In [1]:
import fitz
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

e:\Rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\NAFEES J\AppData\Local\Temp\ipykernel_1092\1753416428.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [3]:
PDF_PATH = r"E:\Rag\data\bis_report_2026.pdf"

doc = fitz.open(PDF_PATH)

print(f"Total Pages: {len(doc)}")

Total Pages: 133


In [4]:
pages = []

for page_num, page in enumerate(doc):
    text = page.get_text("text")

    pages.append(
        Document(
            page_content=text,
            metadata={
                "page": page_num + 1,
                "source": "bis_report_2026.pdf",
                "content_type": "text"
            }
        )
    )

print(f"Extracted {len(pages)} pages")

Extracted 133 pages


In [5]:
def clean_text(text):
    text = re.sub(r'\n{2,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

for page in pages:
    page.page_content = clean_text(page.page_content)

In [6]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = splitter.split_documents(pages)

print(f"Generated {len(chunks)} chunks")

Generated 135 chunks


In [9]:
print(chunks[40].page_content)

BIS Annual Economic Report 2026 
31
 
delivering on it, would underpin that credibility. Securing such credibility would help 
keep risk premia contained and secure fiscal space that otherwise could be eroded. 
Authorities must resist the temptation to seek monetary escape valves. Pressures 
on central banks to accommodate fiscal needs – whether through lower interest rates, 
extended balance sheet support or forbearance on sovereign exposures – may offer 
apparent short-term reliefs, but they undermine the core pillar on which low and 
stable financing costs rest. Past episodes in which fiscal-monetary lines have blurred 
provide unambiguous lessons: the gains are temporary while the costs of restoring 
credibility are large and protracted, with the ultimate burden falling on households 
and firms in the form of higher inflation and macroeconomic instability. 
The latest Middle East conflict has put energy security and supply chain resilience 
to the fore, highlighting the role of str

In [10]:
print(chunks[40].metadata)

{'page': 41, 'source': 'bis_report_2026.pdf', 'content_type': 'text'}


In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

C:\Users\NAFEES J\AppData\Local\Temp\ipykernel_1092\1584869368.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2336.38it/s]


In [13]:
visual_file = Path(r"E:\Rag\data\visual_summaries.md")

visual_text = visual_file.read_text(encoding="utf-8")

# Split each visual entry
visual_sections = re.split(r"(?=## Visual)", visual_text)

visual_docs = []

for section in visual_sections:
    section = section.strip()
    if not section:
        continue

    visual_docs.append(
        Document(
            page_content=section,
            metadata={
                "source": "visual_summaries.md",
                "content_type": "visual"
            }
        )
    )

print(f"Loaded {len(visual_docs)} visual summaries")

Loaded 45 visual summaries


In [14]:
all_documents = chunks + visual_docs

vector_db = Chroma.from_documents(
    documents=all_documents,
    embedding=embedding_model,
    persist_directory="vector_db"
)
vector_db.persist()

C:\Users\NAFEES J\AppData\Local\Temp\ipykernel_1092\815988466.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [15]:
results = vector_db.similarity_search(
    "What are the risks associated with AI?",
    k=5
)

for doc in results:
    print("=" * 80)
    print(doc.metadata)
    print(doc.page_content[:500])

{'content_type': 'visual', 'source': 'visual_summaries.md'}
## Visual ID 13

Chapter: I (Box C)

Figure Number: Graph C1

Figure Title: AI’s long-run impacts are uncertain and scenario-dependent

Visual Type: Line graph and bar chart

Topics:
AI
Output growth
Labour share
Natural rate of interest (r-star)
Automation
Economic scenarios

Description
Found in Box C, this visual presents four long-term scenarios for AI's impact on the economy: Business as usual, Bounded productivity boost, Demand bottleneck, and Transformative AI. Panel A projects trend outp
{'source': 'bis_report_2026.pdf', 'content_type': 'text', 'page': 42}
32 
BIS Annual Economic Report 2026
 
based approaches. These could include imposing minimum haircuts on securities 
financing transactions, tightening liquidity management requirements for open-
ended funds and using central clearing more broadly.5 The data gaps themselves, not 
least in relation to private credit, also represent a vulnerability, as authorities cann